Model should be trained multiple times with different random seeds to get the best model.

In [ ]:
%cd /kaggle/working

!git clone https://github.com/phusroyal/ViHOS.git
%cd /kaggle/working/ViHOS

!find . -maxdepth 3 -type f | sed 's#^\./##' | sort | head -100

In [ ]:
!pip install -q pytorch-crf

# LOAD DATA

In [ ]:
import os
import ast
import random
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchcrf import CRF

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print("CUDA available:", torch.cuda.is_available())


In [ ]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.options.display.max_rows


In [ ]:
REPO_DIR = "/kaggle/working/ViHOS"
DATA_DIR = f"{REPO_DIR}/data"

# Chạy lần 1:
DATA_LEVEL = "syllable"

# Chạy lần 2 thì đổi thành:
# DATA_LEVEL = "word"

if DATA_LEVEL == "syllable":
    train_bio_path = f"{DATA_DIR}/Sequence_labeling_based_version/Syllable/train_BIO_syllable.csv"
    dev_bio_path   = f"{DATA_DIR}/Sequence_labeling_based_version/Syllable/dev_BIO_syllable.csv"
    test_bio_path  = f"{DATA_DIR}/Sequence_labeling_based_version/Syllable/test_BIO_syllable.csv"
    model_display_name = "BiLSTM-CRF + Pho2W_syllable"
else:
    train_bio_path = f"{DATA_DIR}/Sequence_labeling_based_version/Word/train_BIO_Word.csv"
    dev_bio_path   = f"{DATA_DIR}/Sequence_labeling_based_version/Word/dev_BIO_Word.csv"
    test_bio_path  = f"{DATA_DIR}/Sequence_labeling_based_version/Word/test_BIO_Word.csv"
    model_display_name = "BiLSTM-CRF + Pho2W_word"

test_gold_path = f"{DATA_DIR}/Test_data/test.csv"

print("DATA_LEVEL:", DATA_LEVEL)
print("Model:", model_display_name)
print("Train:", train_bio_path)
print("Dev:", dev_bio_path)
print("Test BIO:", test_bio_path)
print("Gold test:", test_gold_path)


# Read BIO data

In [ ]:
label2id = {
    "O": 0,
    "B-HOS": 1,
    "I-HOS": 2,
    "B-T": 1,
    "I-T": 2,
}

id2label = {
    0: "O",
    1: "B-HOS",
    2: "I-HOS",
}


def read_bio(file_path):
    df = pd.read_csv(file_path)
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
    df.columns = [c.strip().lower() for c in df.columns]

    if "word" not in df.columns or "tag" not in df.columns:
        raise ValueError(f"BIO file thiếu word/tag. Columns: {df.columns.tolist()}")

    sentences = []
    labels = []

    if "sentence_id" in df.columns:
        for _, group in df.groupby("sentence_id", sort=False):
            words = group["word"].astype(str).tolist()
            tags = group["tag"].astype(str).tolist()
            sentences.append(words)
            labels.append([label2id[str(t)] for t in tags])
    else:
        cur_words, cur_tags = [], []

        for _, row in df.iterrows():
            if row["index"] == 0 and len(cur_words) > 0:
                sentences.append(cur_words)
                labels.append(cur_tags)
                cur_words, cur_tags = [], []

            cur_words.append(str(row["word"]))
            cur_tags.append(label2id[str(row["tag"])])

        if cur_words:
            sentences.append(cur_words)
            labels.append(cur_tags)

    return sentences, labels


train_sentences, train_labels = read_bio(train_bio_path)
dev_sentences, dev_labels = read_bio(dev_bio_path)
test_sentences, test_labels = read_bio(test_bio_path)

print("Train:", len(train_sentences))
print("Dev:", len(dev_sentences))
print("Test:", len(test_sentences))
print(train_sentences[0][:20])
print(train_labels[0][:20])


# Build vocabulary

In [ ]:
MIN_FREQ = 1
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

counter = Counter()

for sent in train_sentences:
    counter.update([w.lower() for w in sent])

word2id = {
    PAD_TOKEN: 0,
    UNK_TOKEN: 1,
}

for word, freq in counter.items():
    if freq >= MIN_FREQ:
        word2id[word] = len(word2id)

id2word = {i: w for w, i in word2id.items()}

print("Vocab size:", len(word2id))


# Embedding matrix

Nếu có file Pho2W/word2vec 100 chiều trong Kaggle, set `EMBEDDING_PATH` về đường dẫn đó. Nếu không có, notebook dùng random embedding để pipeline vẫn chạy được, nhưng kết quả sẽ không hoàn toàn tương đương paper.

In [ ]:
EMBEDDING_DIM = 100

# Nếu bạn upload/có sẵn embedding, sửa đường dẫn ở đây.
# Ví dụ:
# EMBEDDING_PATH = "/kaggle/input/pho2w/word2vec_vi_words_100dims.txt"
EMBEDDING_PATH = None

# Auto-search một số file embedding phổ biến trong working/input.
candidate_names = [
    "word2vec_vi_words_100dims.txt",
    "word2vec_vi_syllables_100dims.txt",
    "Pho2W_words_100dims.txt",
    "Pho2W_syllables_100dims.txt",
]

if EMBEDDING_PATH is None:
    search_roots = ["/kaggle/working", "/kaggle/input"]
    found_candidates = []
    for root_dir in search_roots:
        if os.path.exists(root_dir):
            for root, dirs, files in os.walk(root_dir):
                for f in files:
                    lower_f = f.lower()
                    if (
                        "100" in lower_f
                        and (
                            "word2vec" in lower_f
                            or "pho2w" in lower_f
                            or "embedding" in lower_f
                        )
                        and f.endswith(".txt")
                    ):
                        found_candidates.append(os.path.join(root, f))
    print("Found embedding candidates:")
    for p in found_candidates[:20]:
        print(" -", p)

    # Không auto chọn để tránh chọn nhầm word/syllable; nếu muốn dùng thì set EMBEDDING_PATH thủ công rồi chạy lại cell.
    if found_candidates:
        print("\nSet EMBEDDING_PATH manually to the correct Pho2W file, then rerun this cell.")


def load_embedding_matrix(embedding_path, word2id, embedding_dim=100):
    matrix = np.random.normal(scale=0.05, size=(len(word2id), embedding_dim)).astype(np.float32)
    matrix[word2id[PAD_TOKEN]] = np.zeros(embedding_dim, dtype=np.float32)

    found = 0

    with open(embedding_path, encoding="utf-8", errors="ignore") as f:
        first_line = f.readline()
        parts = first_line.rstrip().split()

        # Nếu dòng đầu là header dạng: vocab_size dim
        if len(parts) == 2 and all(x.isdigit() for x in parts):
            pass
        else:
            # xử lý luôn dòng đầu nếu không phải header
            if len(parts) == embedding_dim + 1:
                word = parts[0]
                if word.lower() in word2id:
                    matrix[word2id[word.lower()]] = np.asarray(parts[1:], dtype=np.float32)
                    found += 1

        for line in f:
            parts = line.rstrip().split()
            if len(parts) != embedding_dim + 1:
                continue

            word = parts[0]
            key = word.lower()
            if key in word2id:
                matrix[word2id[key]] = np.asarray(parts[1:], dtype=np.float32)
                found += 1

    print("Embedding found:", found, "/", len(word2id))
    return torch.tensor(matrix, dtype=torch.float32)


if EMBEDDING_PATH is not None and os.path.exists(EMBEDDING_PATH):
    print("Loading embedding:", EMBEDDING_PATH)
    embedding_matrix = load_embedding_matrix(EMBEDDING_PATH, word2id, EMBEDDING_DIM)
else:
    print("WARNING: Không tìm/không set EMBEDDING_PATH. Đang dùng random embedding.")
    embedding_matrix = torch.randn(len(word2id), EMBEDDING_DIM) * 0.05
    embedding_matrix[word2id[PAD_TOKEN]] = torch.zeros(EMBEDDING_DIM)

print("Embedding matrix:", embedding_matrix.shape)


# Dataset / DataLoader

In [ ]:
class BIODataset(Dataset):
    def __init__(self, sentences, labels, word2id, max_len=128):
        self.sentences = sentences
        self.labels = labels
        self.word2id = word2id
        self.max_len = max_len

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        words = self.sentences[idx]
        labels = self.labels[idx]

        word_ids = [
            self.word2id.get(w.lower(), self.word2id[UNK_TOKEN])
            for w in words
        ]

        word_ids = word_ids[:self.max_len]
        labels = labels[:self.max_len]
        mask = [1] * len(word_ids)

        pad_len = self.max_len - len(word_ids)

        if pad_len > 0:
            word_ids += [self.word2id[PAD_TOKEN]] * pad_len
            labels += [0] * pad_len
            mask += [0] * pad_len

        return {
            "input_ids": torch.tensor(word_ids, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "mask": torch.tensor(mask, dtype=torch.bool),
            "tokens": words,
        }


def collate_fn(batch):
    return {
        "input_ids": torch.stack([x["input_ids"] for x in batch]),
        "labels": torch.stack([x["labels"] for x in batch]),
        "mask": torch.stack([x["mask"] for x in batch]),
        "tokens": [x["tokens"] for x in batch],
    }


MAX_LEN = 128
BATCH_SIZE = 64

train_loader = DataLoader(
    BIODataset(train_sentences, train_labels, word2id, MAX_LEN),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

dev_loader = DataLoader(
    BIODataset(dev_sentences, dev_labels, word2id, MAX_LEN),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    BIODataset(test_sentences, test_labels, word2id, MAX_LEN),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

batch = next(iter(train_loader))
for k, v in batch.items():
    if k != "tokens":
        print(k, v.shape, v.dtype)
print("tokens sample:", batch["tokens"][0][:10])


# BiLSTM-CRF model

In [ ]:
class BiLSTMCRF(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        num_labels,
        embedding_matrix=None,
        dropout=0.1
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=word2id[PAD_TOKEN]
        )

        if embedding_matrix is not None:
            self.embedding.weight.data.copy_(embedding_matrix)

        self.dropout = nn.Dropout(dropout)

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.classifier = nn.Linear(hidden_dim * 2, num_labels)
        self.crf = CRF(num_labels, batch_first=True)

    def forward(self, input_ids, mask):
        emb = self.embedding(input_ids)
        emb = self.dropout(emb)

        lstm_out, _ = self.lstm(emb)
        lstm_out = self.dropout(lstm_out)

        emissions = self.classifier(lstm_out)
        return emissions

    def loss(self, input_ids, labels, mask):
        emissions = self.forward(input_ids, mask)
        nll = -self.crf(emissions, labels, mask=mask, reduction="mean")
        return nll

    def decode(self, input_ids, mask):
        emissions = self.forward(input_ids, mask)
        return self.crf.decode(emissions, mask=mask)


# Train

In [ ]:
from sklearn.metrics import f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BiLSTMCRF(
    vocab_size=len(word2id),
    embedding_dim=100,
    hidden_dim=60,
    num_labels=3,
    embedding_matrix=embedding_matrix,
    dropout=0.1
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

NUM_EPOCHS = 10


def evaluate_token_f1(model, loader, device):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)
            mask = batch["mask"].to(device)

            decoded = model.decode(input_ids, mask)

            for pred_seq, gold_seq, mask_seq in zip(
                decoded,
                labels.cpu().tolist(),
                mask.cpu().tolist()
            ):
                valid_len = sum(mask_seq)
                all_preds.extend(pred_seq[:valid_len])
                all_labels.extend(gold_seq[:valid_len])

    return f1_score(all_labels, all_preds, average="macro")


for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0.0

    print("Epoch:", epoch + 1)

    for batch in tqdm(train_loader):
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        mask = batch["mask"].to(device)

        optimizer.zero_grad(set_to_none=True)

        loss = model.loss(input_ids, labels, mask)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)
    dev_f1 = evaluate_token_f1(model, dev_loader, device)

    print("Train loss:", train_loss)
    print("Dev token macro F1:", dev_f1)

torch.save(model.state_dict(), f"/kaggle/working/{model_display_name.replace(' ', '_').replace('+', '').replace('/', '_')}.pt")
print("Saved model.")


# Official-style span-level evaluation

In [ ]:
def parse_gold_spans(value):
    if pd.isna(value):
        return set()

    if isinstance(value, str):
        value = value.strip()
        if value == "" or value == "[]":
            return set()
        try:
            value = ast.literal_eval(value)
        except Exception:
            return set()

    result = set()

    def collect(x):
        if isinstance(x, int):
            result.add(x)
        elif isinstance(x, float) and not np.isnan(x):
            result.add(int(x))
        elif isinstance(x, (list, tuple, set)):
            for item in x:
                collect(item)

    collect(value)
    return result


def get_text_and_span_columns(df):
    possible_text_cols = ["content", "text", "comment", "sentence"]
    possible_span_cols = ["index_spans", "span_ids", "spans", "labels"]

    text_col = None
    span_col = None

    for col in possible_text_cols:
        if col in df.columns:
            text_col = col
            break

    for col in possible_span_cols:
        if col in df.columns:
            span_col = col
            break

    if text_col is None or span_col is None:
        raise ValueError(f"Không tìm được cột text/span. Columns: {df.columns.tolist()}")

    return text_col, span_col


def count_contiguous_spans(char_indices):
    if not char_indices:
        return 0

    sorted_idx = sorted(char_indices)
    count = 1

    for i in range(1, len(sorted_idx)):
        if sorted_idx[i] != sorted_idx[i - 1] + 1:
            count += 1

    return count


def prf_from_char_sets(gold_sets, pred_sets):
    tp = 0
    pred_total = 0
    gold_total = 0

    for gold, pred in zip(gold_sets, pred_sets):
        tp += len(gold & pred)
        pred_total += len(pred)
        gold_total += len(gold)

    precision = tp / pred_total if pred_total > 0 else 0.0
    recall = tp / gold_total if gold_total > 0 else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0.0
    )

    return precision, recall, f1


def align_token_to_text(token, text, cursor, data_level):
    token = str(token)

    if data_level == "word":
        candidates = [
            token,
            token.replace("_", " "),
            token.replace("_", ""),
        ]
    else:
        candidates = [token]

    for cand in candidates:
        start = text.find(cand, cursor)
        if start != -1:
            end = start + len(cand)
            return start, end

    for cand in candidates:
        start = text.find(cand)
        if start != -1:
            end = start + len(cand)
            return start, end

    return None


def align_tokens_to_text(tokens, text, data_level):
    offsets = []
    cursor = 0

    for tok in tokens:
        offset = align_token_to_text(tok, text, cursor, data_level)

        if offset is None:
            offsets.append(None)
            continue

        start, end = offset
        offsets.append((start, end))
        cursor = end

    return offsets


def pred_labels_to_char_indices(tokens, pred_labels, text, data_level):
    offsets = align_tokens_to_text(tokens, text, data_level)
    pred_chars = set()

    for label, offset in zip(pred_labels, offsets):
        if offset is None:
            continue

        if label != 0:
            start, end = offset
            pred_chars.update(range(start, end))

    return pred_chars


def predict_bilstm_crf_labels(model, sentences, word2id, device, max_len=128):
    model.eval()
    all_preds = []

    with torch.no_grad():
        for sent in tqdm(sentences):
            word_ids = [
                word2id.get(w.lower(), word2id[UNK_TOKEN])
                for w in sent
            ]

            word_ids = word_ids[:max_len]
            mask = [1] * len(word_ids)

            pad_len = max_len - len(word_ids)

            if pad_len > 0:
                word_ids += [word2id[PAD_TOKEN]] * pad_len
                mask += [0] * pad_len

            input_ids = torch.tensor([word_ids], dtype=torch.long).to(device)
            mask_tensor = torch.tensor([mask], dtype=torch.bool).to(device)

            decoded = model.decode(input_ids, mask_tensor)[0]

            if len(decoded) < len(sent):
                decoded += [0] * (len(sent) - len(decoded))

            all_preds.append(decoded[:len(sent)])

    return all_preds


def evaluate_span_level_bilstm_crf(
    model,
    test_sentences,
    word2id,
    test_gold_path,
    device,
    data_level,
    model_name,
    max_len=128,
    save_csv=True
):
    gold_df = pd.read_csv(test_gold_path)
    gold_df = gold_df.loc[:, ~gold_df.columns.str.contains("^Unnamed")]
    gold_df.columns = [c.strip() for c in gold_df.columns]

    text_col, span_col = get_text_and_span_columns(gold_df)

    gold_texts = gold_df[text_col].astype(str).tolist()
    gold_sets = gold_df[span_col].apply(parse_gold_spans).tolist()

    if len(test_sentences) != len(gold_texts):
        print("WARNING: Số câu BIO và số dòng gold test.csv không khớp!")
        print("BIO sentences:", len(test_sentences))
        print("Gold rows:", len(gold_texts))

        n = min(len(test_sentences), len(gold_texts))
        test_sentences = test_sentences[:n]
        gold_texts = gold_texts[:n]
        gold_sets = gold_sets[:n]

    pred_label_sentences = predict_bilstm_crf_labels(
        model=model,
        sentences=test_sentences,
        word2id=word2id,
        device=device,
        max_len=max_len
    )

    pred_sets = []

    for tokens, pred_labels, text in zip(
        test_sentences,
        pred_label_sentences,
        gold_texts
    ):
        pred_chars = pred_labels_to_char_indices(
            tokens=tokens,
            pred_labels=pred_labels,
            text=text,
            data_level=data_level
        )
        pred_sets.append(pred_chars)

    single_indices = [
        i for i, gold in enumerate(gold_sets)
        if count_contiguous_spans(gold) == 1
    ]

    multiple_indices = [
        i for i, gold in enumerate(gold_sets)
        if count_contiguous_spans(gold) > 1
    ]

    all_indices = [
        i for i, gold in enumerate(gold_sets)
        if count_contiguous_spans(gold) >= 1
    ]

    def subset_score(indices):
        subset_gold = [gold_sets[i] for i in indices]
        subset_pred = [pred_sets[i] for i in indices]
        return prf_from_char_sets(subset_gold, subset_pred)

    single_p, single_r, single_f1 = subset_score(single_indices)
    multiple_p, multiple_r, multiple_f1 = subset_score(multiple_indices)
    all_p, all_r, all_f1 = subset_score(all_indices)

    result = {
        "model": model_name,

        "single_precision": single_p,
        "single_recall": single_r,
        "single_f1": single_f1,

        "multiple_precision": multiple_p,
        "multiple_recall": multiple_r,
        "multiple_f1": multiple_f1,

        "all_precision": all_p,
        "all_recall": all_r,
        "all_f1": all_f1,

        "n_single": len(single_indices),
        "n_multiple": len(multiple_indices),
        "n_all": len(all_indices),
    }

    print("===== BiLSTM-CRF span-level official-style benchmark =====")
    for k, v in result.items():
        if k == "model":
            print(f"{k}: {v}")
        elif k.startswith("n_"):
            print(f"{k}: {v}")
        else:
            print(f"{k}: {v:.6f}")

    if save_csv:
        result_path = Path("/kaggle/working/bilstm_crf_span_level_results.csv")

        if result_path.exists():
            old_df = pd.read_csv(result_path)
            new_df = pd.concat([old_df, pd.DataFrame([result])], ignore_index=True)
        else:
            new_df = pd.DataFrame([result])

        new_df.to_csv(result_path, index=False)
        print(f"\nSaved result to: {result_path}")

    return result, pred_sets, gold_sets


# Run benchmark

In [ ]:
span_result, pred_spans, gold_spans = evaluate_span_level_bilstm_crf(
    model=model,
    test_sentences=test_sentences,
    word2id=word2id,
    test_gold_path=test_gold_path,
    device=device,
    data_level=DATA_LEVEL,
    model_name=model_display_name,
    max_len=MAX_LEN,
    save_csv=True
)


In [ ]:
pd.read_csv("/kaggle/working/bilstm_crf_span_level_results.csv")

# Notes

- Để chạy biến thể còn lại, đổi `DATA_LEVEL` ở cell config path:
  - `syllable` → `BiLSTM-CRF + Pho2W_syllable`
  - `word` → `BiLSTM-CRF + Pho2W_word`
- Nếu muốn gần paper hơn, cần set `EMBEDDING_PATH` tới đúng file Pho2W 100 chiều trước khi train.
- File kết quả benchmark sẽ được append vào `/kaggle/working/bilstm_crf_span_level_results.csv`.
